## Q18-22

In [12]:
# ================================================================
# ECE 232E Project 2 — Part 2: Google+ Network
# Q18, Q19, Q20, Q22  (Q21 is a written-answer question)
#
# Corrections vs original:
#   1. build_nx_digraph / build_igraph_undirected now add the ego
#      node and directed edges  ego → every neighbor  (matching
#      the reference R code: add.vertices + add.edges loop).
#   2. Q22 compute_h_c: b_i counts ONLY labelled nodes in each
#      community (reference had a bug: it counted all community
#      members including unlabelled ones).
#   3. Q19 linear-scale plot changed to bar chart (closer to the
#      reference hist() style).
# ================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import igraph as ig
from collections import Counter

# ── Paths (change BASE_DIR to your working directory) ────────────
BASE_DIR  = "."
GPLUS_DIR = os.path.join(BASE_DIR, "gplus")
IMAGE_DIR = os.path.join(BASE_DIR, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

TARGET_EGO_IDS = [
    "109327480479767108490",
    "115625564993990145546",
    "101373961279443806744",
]

# ════════════════════════════════════════════════════════════════
# Shared helpers
# ════════════════════════════════════════════════════════════════

def count_circles(circles_path):
    """Count non-empty lines (= circles) in a .circles file."""
    with open(circles_path, "r") as f:
        return sum(1 for line in f if line.strip())


def parse_circles(ego_id):
    """
    Parse <ego_id>.circles.

    Returns
    -------
    node_to_circle : dict  {node_name -> circle_idx}
        Each node is assigned to its FIRST-APPEARING circle only
        (handles overlapping circles with a single-label rule).
    circle_names : list[str]
        Ordered circle label strings.
    """
    path = os.path.join(GPLUS_DIR, f"{ego_id}.circles")
    node_to_circle, circle_names = {}, []
    with open(path, "r") as f:
        for c_idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            circle_names.append(parts[0])
            for m in parts[1:]:
                m = m.strip()
                if m and m not in node_to_circle:   # first-occurrence wins
                    node_to_circle[m] = c_idx
    return node_to_circle, circle_names


# ── FIX 1: ego node + directed ego→neighbor edges ───────────────

def build_nx_digraph(ego_id):
    """
    Build a directed NetworkX graph from <ego_id>.edges, then add
    the ego node itself with a directed edge  ego → v  for every
    node v that already exists in the graph.

    This matches the reference R code:
        final_graph[[i]] = add.vertices(initial_graph[[i]], 1, name=node)
        for k in 1:length(graph_nodes):
            add.edges(final_graph[[i]], c(core_index, k))
    """
    edges_path = os.path.join(GPLUS_DIR, f"{ego_id}.edges")
    G = nx.DiGraph()
    with open(edges_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                G.add_edge(parts[0], parts[1])

    # Snapshot of nodes BEFORE adding ego, then add ego → each
    existing_nodes = list(G.nodes())
    G.add_node(ego_id)
    for node in existing_nodes:
        G.add_edge(ego_id, node)

    return G


def build_igraph_undirected(ego_id):
    """
    Build an igraph UNDIRECTED graph from <ego_id>.edges, then add
    the ego node with edges  ego ↔ every other node.

    Walktrap requires an undirected graph; converting after adding
    the ego node (with directed ego→neighbor edges) is equivalent
    to the reference which calls cluster_walktrap on a directed
    graph — igraph treats it as undirected internally.

    Returns
    -------
    G_undir  : ig.Graph  (undirected)
    name2idx : dict  {node_name -> vertex index}
    """
    edges_path = os.path.join(GPLUS_DIR, f"{ego_id}.edges")
    edge_list, node_set = [], set()
    with open(edges_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                edge_list.append((parts[0], parts[1]))
                node_set.update(parts[:2])

    # Keep original nodes for ego-edge construction
    original_nodes = sorted(node_set)

    # Add ego node to the universe
    node_set.add(ego_id)
    nodes    = sorted(node_set)
    name2idx = {n: i for i, n in enumerate(nodes)}

    # Directed edge list: original edges + ego → each original node
    idx_edges = [(name2idx[u], name2idx[v]) for u, v in edge_list]
    ego_idx   = name2idx[ego_id]
    for node in original_nodes:
        idx_edges.append((ego_idx, name2idx[node]))

    G_dir = ig.Graph(n=len(nodes), edges=idx_edges, directed=True)
    G_dir.vs["name"] = nodes
    G_undir = G_dir.as_undirected(combine_edges="first")
    return G_undir, name2idx


# ════════════════════════════════════════════════════════════════
# Q18 — How many personal networks?
# ════════════════════════════════════════════════════════════════
print("=" * 60)
print("Q18 — Number of Personal Networks")
print("=" * 60)

all_circles_files = [f for f in os.listdir(GPLUS_DIR) if f.endswith(".circles")]
all_ego_ids       = [f.replace(".circles", "") for f in all_circles_files]

valid_ego_ids = []
for ego_id in sorted(all_ego_ids):
    edges_path   = os.path.join(GPLUS_DIR, f"{ego_id}.edges")
    circles_path = os.path.join(GPLUS_DIR, f"{ego_id}.circles")
    if not os.path.exists(edges_path):
        continue
    if count_circles(circles_path) > 2:
        valid_ego_ids.append(ego_id)

print(f"Total ego nodes with .circles file : {len(all_ego_ids)}")
print(f"Personal networks with > 2 circles : {len(valid_ego_ids)}")
print(f"\n→ Answer Q18: {len(valid_ego_ids)} personal networks")


# ════════════════════════════════════════════════════════════════
# Q19 — In-degree / Out-degree Distributions
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Q19 — In/Out-degree Distributions")
print("=" * 60)


def plot_degree_dist(ego_id, in_degrees, out_degrees):
    """
    For each of in-degree and out-degree, draw two side-by-side plots:
      left  — bar chart on linear scale (matches reference hist())
      right — scatter on log-log scale
    Saves to IMAGE_DIR.
    """
    for degrees, label, fname_tag, color in [
        (in_degrees,  "In-degree",  "indegree",  "steelblue"),
        (out_degrees, "Out-degree", "outdegree", "tomato"),
    ]:
        count = Counter(degrees)
        xs    = sorted(count.keys())
        ys    = [count[k] for k in xs]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        fig.suptitle(
            f"{label} Distribution — Ego: …{ego_id[-6:]}",
            fontsize=11,
        )

        # Linear bar chart
        ax1.bar(xs, ys, color=color, alpha=0.8, width=max(xs) * 0.005 + 1)
        ax1.set_xlabel(label)
        ax1.set_ylabel("Number of nodes")
        ax1.set_title("Linear scale")
        ax1.grid(True, linestyle="--", alpha=0.4)

        # Log-log scatter
        xs_p = [x for x, y in zip(xs, ys) if x > 0 and y > 0]
        ys_p = [y for x, y in zip(xs, ys) if x > 0 and y > 0]
        ax2.scatter(xs_p, ys_p, s=20, color=color, alpha=0.85)
        ax2.set_xscale("log")
        ax2.set_yscale("log")
        ax2.set_xlabel(f"{label} (log)")
        ax2.set_ylabel("Number of nodes (log)")
        ax2.set_title("Log-log scale")
        ax2.grid(True, which="both", linestyle="--", alpha=0.4)

        plt.tight_layout()
        path = os.path.join(IMAGE_DIR, f"q19_{ego_id}_{fname_tag}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {path}")


print(f"\n{'Ego Node (last 20)':<28} {'Nodes':>7} {'Edges':>10} "
      f"{'Avg In':>8} {'Max In':>7} {'Avg Out':>8} {'Max Out':>8}")
print("-" * 82)

for ego_id in TARGET_EGO_IDS:
    G       = build_nx_digraph(ego_id)
    in_deg  = [d for _, d in G.in_degree()]
    out_deg = [d for _, d in G.out_degree()]

    print(f"  …{ego_id[-20:]:<26} {G.number_of_nodes():>7} "
          f"{G.number_of_edges():>10} "
          f"{np.mean(in_deg):>8.2f} {max(in_deg):>7} "
          f"{np.mean(out_deg):>8.2f} {max(out_deg):>8}")

    plot_degree_dist(ego_id, in_deg, out_deg)

print("\nNote: ego node has out-degree = (# nodes in network - 1) "
      "and in-degree = 0 unless neighbors link back.")


# ════════════════════════════════════════════════════════════════
# Q20 — Walktrap Community Detection
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Q20 — Walktrap Community Detection")
print("=" * 60)

# Results stored here for reuse in Q22
q20_results = {}   # ego_id → {membership, name2idx, modularity, n_comm}


def plot_community_walktrap(G_undir, membership, ego_id,
                            modularity, n_communities, save_path):
    n    = G_undir.vcount()
    cmap = plt.colormaps.get_cmap("tab20")

    def get_color(c):
        return cmap(c / max(n_communities - 1, 1))

    colors = [get_color(membership[v]) for v in range(n)]
    v_size = 5 if n < 1000 else 3

    # Choose layout by graph size
    if n < 500:
        layout = G_undir.layout("fr")
    elif n < 2000:
        layout = G_undir.layout("fr", niter=150)
    else:
        layout = G_undir.layout("drl")

    coords = np.array(layout.coords)
    pad_x  = coords[:, 0].ptp() * 0.05
    pad_y  = coords[:, 1].ptp() * 0.05

    fig, ax = plt.subplots(figsize=(11, 10))
    ig.plot(
        G_undir, target=ax, layout=layout,
        vertex_color=colors, vertex_size=v_size,
        vertex_frame_width=0.2, edge_width=0.15, edge_color="#cccccc",
    )
    ax.set_xlim(coords[:, 0].min() - pad_x, coords[:, 0].max() + pad_x)
    ax.set_ylim(coords[:, 1].min() - pad_y, coords[:, 1].max() + pad_y)
    ax.set_title(
        f"Walktrap Community Structure\n"
        f"Ego: …{ego_id[-6:]}  |  "
        f"Communities: {n_communities}  |  "
        f"Modularity: {modularity:.4f}",
        fontsize=12,
    )

    shown = min(n_communities, 18)
    patches = [
        mpatches.Patch(color=get_color(i), label=f"Community {i + 1}")
        for i in range(shown)
    ]
    if n_communities > shown:
        patches.append(mpatches.Patch(
            color="white", label=f"… (+{n_communities - shown} more)"))
    ax.legend(handles=patches, loc="upper right",
              fontsize=7, ncol=2, framealpha=0.8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {save_path}")


for ego_id in TARGET_EGO_IDS:
    print(f"\nEgo node: {ego_id}")
    G_undir, name2idx = build_igraph_undirected(ego_id)
    print(f"  Nodes (undirected): {G_undir.vcount()}")
    print(f"  Edges (undirected): {G_undir.ecount()}")

    walktrap    = G_undir.community_walktrap(steps=4)
    communities = walktrap.as_clustering()
    membership  = communities.membership
    modularity  = communities.modularity
    n_comm      = len(communities)
    sizes       = sorted([len(c) for c in communities], reverse=True)

    print(f"  Communities : {n_comm}")
    print(f"  Modularity  : {modularity:.6f}")
    print(f"  Sizes (top5): {sizes[:5]}")

    q20_results[ego_id] = {
        "membership": membership,
        "name2idx":   name2idx,
        "modularity": modularity,
        "n_comm":     n_comm,
    }

    save_path = os.path.join(IMAGE_DIR, f"q20_{ego_id}_walktrap.png")
    plot_community_walktrap(G_undir, membership, ego_id,
                            modularity, n_comm, save_path)

print("\nQ20 Summary:")
print(f"{'Ego Node':<32} {'Communities':>13} {'Modularity':>12}")
print("-" * 60)
for ego_id in TARGET_EGO_IDS:
    r = q20_results[ego_id]
    print(f"  …{ego_id[-20:]:<30} {r['n_comm']:>13} {r['modularity']:>12.6f}")


# ════════════════════════════════════════════════════════════════
# Q22 — Homogeneity (h) and Completeness (c)
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("Q22 — Homogeneity and Completeness")
print("=" * 60)


def safe_xlogx(x):
    """x * log(x), returning 0 when x == 0 (0 * log(0) := 0)."""
    return x * np.log(x) if x > 0 else 0.0


def compute_h_c(membership, name2idx, node_to_circle):
    """
    Compute homogeneity h and completeness c per the project formulas.

    Parameters
    ----------
    membership     : list[int]   vertex index → community index
    name2idx       : dict        node_name → vertex index
    node_to_circle : dict        node_name → circle index
                                 (first-occurrence, from parse_circles)

    Design choices
    --------------
    - Only nodes that are BOTH in the graph AND in circles are used.
    - N  = number of such unique labelled nodes
    - a  = circle  size vector among labelled nodes  (sums to N)
    - b  = community size vector among labelled nodes (sums to N)
             ← FIX: reference R code used ALL community members here,
               which is inconsistent with N.  We use labelled-only.
    - A[j, i] = # labelled nodes in community j AND circle i
    """
    labelled = [n for n in node_to_circle if n in name2idx]
    if not labelled:
        return None, None, None, None, None, None

    N = len(labelled)
    circles_vec     = [node_to_circle[n]       for n in labelled]
    communities_vec = [membership[name2idx[n]] for n in labelled]

    n_C = max(circles_vec)     + 1
    n_K = max(communities_vec) + 1

    # a[i] = # labelled nodes in circle i
    a = np.zeros(n_C, dtype=float)
    for ci in circles_vec:
        a[ci] += 1

    # b[j] = # labelled nodes in community j  ← CORRECTED
    b = np.zeros(n_K, dtype=float)
    for ki in communities_vec:
        b[ki] += 1

    # A[j, i] = # labelled nodes in community j AND circle i
    A = np.zeros((n_K, n_C), dtype=float)
    for ci, ki in zip(circles_vec, communities_vec):
        A[ki, ci] += 1

    # H(C)
    H_C = -sum(safe_xlogx(ai / N) for ai in a)

    # H(K)
    H_K = -sum(safe_xlogx(bi / N) for bi in b)

    # H(C | K)
    H_CK = 0.0
    for j in range(n_K):
        for i in range(n_C):
            if A[j, i] > 0 and b[j] > 0:
                H_CK -= (A[j, i] / N) * np.log(A[j, i] / b[j])

    # H(K | C)
    H_KC = 0.0
    for i in range(n_C):
        for j in range(n_K):
            if A[j, i] > 0 and a[i] > 0:
                H_KC -= (A[j, i] / N) * np.log(A[j, i] / a[i])

    h = (1.0 - H_CK / H_C) if H_C > 0.0 else 0.0
    c = (1.0 - H_KC / H_K) if H_K > 0.0 else 0.0

    return h, c, H_C, H_K, H_CK, H_KC


print(f"\n{'Ego Node':<32} {'Circles':>8} {'Labelled':>10} "
      f"{'h':>10} {'c':>10}")
print("-" * 74)

for ego_id in TARGET_EGO_IDS:
    print(f"\nEgo node: {ego_id}")
    res        = q20_results[ego_id]
    membership = res["membership"]
    name2idx   = res["name2idx"]

    node_to_circle, circle_names = parse_circles(ego_id)
    labelled = [n for n in node_to_circle if n in name2idx]

    print(f"  Circles                : {len(circle_names)}")
    print(f"  Nodes with circle info : {len(labelled)} / {len(name2idx)}")
    print(f"  Communities (Walktrap) : {res['n_comm']}")

    h, c, H_C, H_K, H_CK, H_KC = compute_h_c(
        membership, name2idx, node_to_circle
    )

    if h is None:
        print("  [SKIP] No labelled nodes found in graph.")
        continue

    print(f"  H(C)   = {H_C:.6f}")
    print(f"  H(K)   = {H_K:.6f}")
    print(f"  H(C|K) = {H_CK:.6f}")
    print(f"  H(K|C) = {H_KC:.6f}")
    print(f"  h      = {h:.6f}")
    print(f"  c      = {c:.6f}")

    if h < 0:
        print("  [NOTE] h < 0: H(C|K) > H(C) — each community contains "
              "members from MORE circles than the overall distribution, "
              "i.e. communities are more mixed than circles.")
    if c < 0:
        print("  [NOTE] c < 0: H(K|C) > H(K) — each circle's members "
              "are spread across MORE communities than the overall "
              "distribution, i.e. circles are more fragmented than "
              "communities.")

print("\n===== DONE Q18–Q22 =====")

Q18 — Number of Personal Networks
Total ego nodes with .circles file : 132
Personal networks with > 2 circles : 57

→ Answer Q18: 57 personal networks

Q19 — In/Out-degree Distributions

Ego Node (last 20)             Nodes      Edges   Avg In  Max In  Avg Out  Max Out
----------------------------------------------------------------------------------
  …09327480479767108490           774      10884    14.06      91    14.06      773
  Saved: ./images/q19_109327480479767108490_indegree.png
  Saved: ./images/q19_109327480479767108490_outdegree.png
  …15625564993990145546           924      40323    43.64     234    43.64      923
  Saved: ./images/q19_115625564993990145546_indegree.png
  Saved: ./images/q19_115625564993990145546_outdegree.png
  …01373961279443806744          3815    1137321   298.12    1826   298.12     3814
  Saved: ./images/q19_101373961279443806744_indegree.png
  Saved: ./images/q19_101373961279443806744_outdegree.png

Note: ego node has out-degree = (# nodes in netwo

## Q23-25

In [14]:
# ================================================================
# ECE 232E Project 2 — Part 3: Cora Dataset
# Q23: GCN  |  Q24: Node2Vec  |  Q25: Personalized PageRank
#
# Corrections vs original:
#   Q23 — Expanded grid (22 configs); patience 50→100; explicit
#          n_layers printed per config; wd grid added.
#   Q24 — get_emb() handles str/int key mismatch across node2vec
#          library versions; combined features tested on all 3
#          classifiers; winner reasoning printed.
#   Q25 — one_walk() returns numpy array (faster); visit counting
#          uses np.unique batching (10-20× faster than per-step
#          dict update); seeds passed as numpy array for speed;
#          zero-visit fallback to majority class documented;
#          per-class F1 report added for every α.
# ================================================================

import os, random, time
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report)
from sklearn.preprocessing import StandardScaler

import networkx as nx
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_networkx
from node2vec import Node2Vec

# ── Paths & global seeds ─────────────────────────────────────────
BASE_DIR = "."           # ← change if running locally
DATA_DIR = os.path.join(BASE_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

SEED = 42
random.seed(SEED);  np.random.seed(SEED);  torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

# ── Load Cora ────────────────────────────────────────────────────
dataset    = Planetoid(root=DATA_DIR, name="Cora")
data       = dataset[0].to(DEVICE)
labels_np  = data.y.cpu().numpy()          # int [2708]
text_np    = data.x.cpu().numpy()          # float [2708, 1433]
train_mask = data.train_mask.cpu().numpy() # bool  [2708]
val_mask   = data.val_mask.cpu().numpy()
test_mask  = data.test_mask.cpu().numpy()

print(f"\nCora dataset:")
print(f"  Nodes           : {data.num_nodes}")
print(f"  Edges           : {data.num_edges}")
print(f"  Features/node   : {data.num_node_features}")
print(f"  Classes         : {dataset.num_classes}")
print(f"  Train/Val/Test  : "
      f"{train_mask.sum()} / {val_mask.sum()} / {test_mask.sum()}")


# ════════════════════════════════════════════════════════════════
# Q23 — Graph Convolutional Network
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*68)
print("Q23 — Graph Convolutional Network (GCN)")
print("═"*68)

class GCN(nn.Module):
    """
    Flexible GCN following Kipf & Welling (ICLR 2017).
    hidden_dims controls both depth and width:
      len(hidden_dims) == 1  →  2-layer network  (1 hidden layer)
      len(hidden_dims) == 2  →  3-layer network
      len(hidden_dims) == 3  →  4-layer network
    Activation: ReLU between layers; log-softmax at output.
    """
    def __init__(self, in_dim, hidden_dims, out_dim, dropout):
        super().__init__()
        self.dropout = dropout
        dims  = [in_dim] + list(hidden_dims) + [out_dim]
        self.convs = nn.ModuleList(
            [GCNConv(dims[i], dims[i+1]) for i in range(len(dims)-1)]
        )

    @property
    def n_layers(self):
        return len(self.convs)

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


def run_gcn(params, data, dataset,
            epochs=500, patience=100, seed=SEED):
    """Train one GCN config; return (val_acc, test_acc, state, n_layers)."""
    torch.manual_seed(seed)
    model = GCN(
        in_dim      = dataset.num_node_features,
        hidden_dims = params["hidden_dims"],
        out_dim     = dataset.num_classes,
        dropout     = params["dropout"],
    ).to(DEVICE)
    opt = torch.optim.Adam(
        model.parameters(), lr=params["lr"], weight_decay=params["wd"]
    )

    best_val = best_te = 0.0
    best_state = None
    no_improve = 0

    for _ in range(1, epochs + 1):
        # ── train step ──────────────────────────────────────────
        model.train(); opt.zero_grad()
        out  = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); opt.step()

        # ── evaluate ────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            out  = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            val_acc  = (pred[data.val_mask]
                        == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask]
                        == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val:
            best_val   = val_acc
            best_te    = test_acc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    return best_val, best_te, best_state, model.n_layers


# ── Hyperparameter grid ──────────────────────────────────────────
#
# Axes explored:
#   n_layers  ∈ {2 (1 hidden), 3 (2 hidden), 4 (3 hidden)}
#   width     ∈ {16, 64, 128, 256}
#   dropout   ∈ {0.3, 0.5}
#   lr        ∈ {0.005, 0.01}
#   wd        ∈ {5e-4, 1e-3}
#
# The 2-layer [16] config is the original Kipf & Welling baseline.
param_grid = [
    # ── 2-layer (1 hidden) ──────────────────────────────────────
    dict(hidden_dims=[16],        dropout=0.5, lr=0.01,  wd=5e-4),  # baseline
    dict(hidden_dims=[64],        dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[128],       dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[256],       dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64],        dropout=0.3, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[128],       dropout=0.3, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64],        dropout=0.5, lr=0.005, wd=5e-4),
    dict(hidden_dims=[128],       dropout=0.5, lr=0.005, wd=5e-4),
    dict(hidden_dims=[64],        dropout=0.5, lr=0.01,  wd=1e-3),
    dict(hidden_dims=[128],       dropout=0.5, lr=0.01,  wd=1e-3),
    # ── 3-layer (2 hidden) ──────────────────────────────────────
    dict(hidden_dims=[64, 32],    dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64, 64],    dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[128, 64],   dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[128, 64],   dropout=0.3, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64, 64],    dropout=0.3, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64, 64],    dropout=0.5, lr=0.005, wd=5e-4),
    dict(hidden_dims=[128, 64],   dropout=0.5, lr=0.005, wd=5e-4),
    dict(hidden_dims=[64, 64],    dropout=0.5, lr=0.01,  wd=1e-3),
    # ── 4-layer (3 hidden) ──────────────────────────────────────
    dict(hidden_dims=[64, 64, 32],  dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64, 64, 64],  dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[128, 64, 32], dropout=0.5, lr=0.01,  wd=5e-4),
    dict(hidden_dims=[64, 64, 32],  dropout=0.3, lr=0.01,  wd=5e-4),
]

EPOCHS   = 500
PATIENCE = 100

print(f"\nGrid search: {len(param_grid)} configurations, "
      f"up to {EPOCHS} epochs, early-stop patience = {PATIENCE}\n")
print(f"  {'Architecture':<30} {'L':>2} {'drop':>5} {'lr':>7} "
      f"{'wd':>7} {'val':>7} {'test':>7}")
print("  " + "─"*72)

gcn_records = []
for params in param_grid:
    val_acc, test_acc, state, n_layers = run_gcn(
        params, data, dataset, EPOCHS, PATIENCE
    )
    arch_inner = "→".join(str(h) for h in params["hidden_dims"])
    arch_str   = f"[1433→{arch_inner}→7]"
    print(f"  {arch_str:<30} {n_layers:>2} {params['dropout']:>5.1f} "
          f"{params['lr']:>7.4f} {params['wd']:>7.5f} "
          f"{val_acc:>7.4f} {test_acc:>7.4f}")
    gcn_records.append({
        **params,
        "val": val_acc, "test": test_acc,
        "n_layers": n_layers, "state": state, "arch": arch_str,
    })

best_rec = max(gcn_records, key=lambda r: r["val"])

# Restore and evaluate best model
torch.manual_seed(SEED)
best_gcn = GCN(
    in_dim      = dataset.num_node_features,
    hidden_dims = best_rec["hidden_dims"],
    out_dim     = dataset.num_classes,
    dropout     = best_rec["dropout"],
).to(DEVICE)
best_gcn.load_state_dict(best_rec["state"])
best_gcn.eval()
with torch.no_grad():
    gcn_logits = best_gcn(data.x, data.edge_index)
    gcn_preds  = gcn_logits.argmax(dim=1)[data.test_mask].cpu().numpy()
gcn_true = data.y[data.test_mask].cpu().numpy()

print(f"\n{'─'*72}")
print(f"Best architecture  : {best_rec['arch']}")
print(f"Number of layers   : {best_rec['n_layers']}")
print(f"Hyperparameters    : dropout={best_rec['dropout']}, "
      f"lr={best_rec['lr']}, wd={best_rec['wd']}")
print(f"Best val  accuracy : {best_rec['val']:.4f}")
print(f"Best test accuracy : {best_rec['test']:.4f}  "
      f"({best_rec['test']*100:.2f}%)")
print(f"\nClassification report (best GCN — test set):")
CLASS_NAMES = ["Case_Based", "Genetic_Alg", "Neural_Net",
               "Prob_Method", "Reinf_Learn", "Rule_Learn", "Theory"]
print(classification_report(gcn_true, gcn_preds,
                             target_names=CLASS_NAMES, digits=4))


# ════════════════════════════════════════════════════════════════
# Q24 — Node2Vec + Classifier
# ════════════════════════════════════════════════════════════════
print("\n" + "═"*68)
print("Q24 — Node2Vec + Classifier")
print("═"*68)

# ── Build undirected NetworkX graph ──────────────────────────────
G_nx = to_networkx(data, to_undirected=True)
print(f"\nGraph (full): {G_nx.number_of_nodes()} nodes, "
      f"{G_nx.number_of_edges()} edges")

# ── Fit Node2Vec ─────────────────────────────────────────────────
#
# Node2Vec (Grover & Leskovec, KDD 2016) generates node embeddings
# via biased second-order random walks parameterised by:
#   p  (return parameter)  — controls likelihood of revisiting a node
#   q  (in-out parameter)  — interpolates BFS (q<1, community) and
#                             DFS (q>1, structural role) exploration
# Walk sequences are treated as "sentences" and fed to Word2Vec
# Skip-gram, so co-occurring nodes in walks get similar embeddings.
# p=q=1 reduces to DeepWalk (unbiased random walk), a strong baseline.

print("\nFitting Node2Vec  (p=1, q=1, dim=128, walk_length=80, "
      "num_walks=10, window=10) ...")
t0 = time.time()
n2v_obj = Node2Vec(
    G_nx,
    dimensions  = 128,
    walk_length = 80,
    num_walks   = 10,
    p           = 1.0,
    q           = 1.0,
    workers     = 4,
    seed        = SEED,
    quiet       = True,
)
n2v_wv = n2v_obj.fit(
    window=10, min_count=1, batch_words=4, epochs=10, seed=SEED
).wv
print(f"  Done in {time.time()-t0:.1f}s")

# Handle str/int key difference across node2vec library versions
def get_emb(wv, node_id):
    """Try string key first, then integer key."""
    try:
        return wv[str(node_id)]
    except KeyError:
        return wv[node_id]

emb = np.array([get_emb(n2v_wv, i) for i in range(data.num_nodes)])
print(f"  Embedding matrix: {emb.shape}  (nodes × dim)")

# ── Feature sets ─────────────────────────────────────────────────
y_tr = labels_np[train_mask]
y_te = labels_np[test_mask]

feat_sets = {
    "Node2Vec only  (128-d)": (
        emb[train_mask],
        emb[test_mask],
    ),
    "Text only     (1433-d)": (
        text_np[train_mask],
        text_np[test_mask],
    ),
    "Combined      (1561-d)": (
        np.hstack([emb[train_mask], text_np[train_mask]]),
        np.hstack([emb[test_mask],  text_np[test_mask]]),
    ),
}

# ── Classifiers ──────────────────────────────────────────────────
def make_clfs():
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=500, max_depth=None,
            random_state=SEED, n_jobs=-1,
        ),
        "SVM (RBF)": SVC(
            C=10, gamma="scale", kernel="rbf",
            random_state=SEED,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(256, 128), max_iter=500,
            random_state=SEED, early_stopping=True,
            validation_fraction=0.1, n_iter_no_change=20,
        ),
    }

print("\nResults — Accuracy / Macro-F1:\n")
print(f"  {'Feature Set':<28} {'Classifier':<16} "
      f"{'Acc':>7} {'Macro-F1':>10}")
print("  " + "─"*64)

q24_results = {}
for feat_label, (X_tr, X_te) in feat_sets.items():
    scaler   = StandardScaler().fit(X_tr)
    X_tr_sc  = scaler.transform(X_tr)
    X_te_sc  = scaler.transform(X_te)

    for clf_name, clf in make_clfs().items():
        needs_scale = clf_name in ("SVM (RBF)", "MLP")
        clf.fit(X_tr_sc if needs_scale else X_tr, y_tr)
        pred = clf.predict(X_te_sc if needs_scale else X_te)

        acc = accuracy_score(y_te, pred)
        f1  = f1_score(y_te, pred, average="macro")
        print(f"  {feat_label:<28} {clf_name:<16} {acc:>7.4f} {f1:>10.4f}")
        q24_results[(feat_label, clf_name)] = (acc, f1)
    print()

# ── Summary ──────────────────────────────────────────────────────
best_key  = max(q24_results, key=lambda k: q24_results[k][0])
best_acc  = q24_results[best_key][0]
best_f1   = q24_results[best_key][1]

acc_n2v  = max(v[0] for k, v in q24_results.items() if "Node2Vec" in k[0])
acc_text = max(v[0] for k, v in q24_results.items() if "Text"     in k[0])
acc_comb = max(v[0] for k, v in q24_results.items() if "Combined" in k[0])

print("─"*68)
print(f"Best acc — Node2Vec only : {acc_n2v:.4f}  ({acc_n2v*100:.2f}%)")
print(f"Best acc — Text only     : {acc_text:.4f}  ({acc_text*100:.2f}%)")
winner = "Text" if acc_text >= acc_n2v else "Node2Vec"
print(f"Winner (single feature)  : {winner}")
print(f"\nBest combined features   : {acc_comb:.4f}  ({acc_comb*100:.2f}%)")
print(f"Best overall (Q24)       : {best_acc:.4f}  ({best_acc*100:.2f}%)")
print(f"  Feature set  : {best_key[0].strip()}")
print(f"  Classifier   : {best_key[1]}")
print(f"  Macro-F1     : {best_f1:.4f}")

print(f"""
Why does Text outperform Node2Vec on Cora?
  Each Cora node has 1433 binary word-occurrence features that
  directly encode the paper's research topic — these features are
  highly discriminative for the 7-class classification task.
  Node2Vec captures only citation graph topology, which is a
  weaker proxy for topic because papers frequently cite across
  research areas. As a result, the 1433-d bag-of-words feature
  vector carries far more class-relevant signal than the 128-d
  structural embedding.

  Combining both feature sets improves accuracy further: Node2Vec
  captures neighbourhood structure (e.g., papers surrounded by
  many Neural-Network papers are likely in that class) which is
  complementary to but not redundant with the text content.
""")


# ════════════════════════════════════════════════════════════════
# Q25 — Personalized PageRank (biased random walk on GCC)
# ════════════════════════════════════════════════════════════════
print("═"*68)
print("Q25 — Personalized PageRank (Biased Random Walk on GCC)")
print("═"*68)

# ── GCC ─────────────────────────────────────────────────────────
gcc_set   = max(nx.connected_components(G_nx), key=len)
G_gcc     = G_nx.subgraph(gcc_set).copy()
gcc_nodes = sorted(gcc_set)

gcc_train = [n for n in gcc_nodes if train_mask[n]]
gcc_test  = [n for n in gcc_nodes if test_mask[n]]

print(f"\nFull graph : {G_nx.number_of_nodes()} nodes / "
      f"{G_nx.number_of_edges()} edges")
print(f"GCC        : {len(gcc_nodes)} nodes / "
      f"{G_gcc.number_of_edges()} edges")
print(f"GCC train  : {len(gcc_train)} / {train_mask.sum()} total")
print(f"GCC test   : {len(gcc_test)} / {test_mask.sum()} total")

# ── Seeds per class ──────────────────────────────────────────────
n_classes      = dataset.num_classes
seeds_by_class = defaultdict(list)
for node in gcc_train:
    seeds_by_class[labels_np[node]].append(node)

print("\nSeeds per class (GCC training nodes):")
for c in range(n_classes):
    print(f"  class {c} [{CLASS_NAMES[c]:>12s}] : "
          f"{len(seeds_by_class[c])} seeds")

majority_class = Counter(
    labels_np[n] for n in gcc_train
).most_common(1)[0][0]

# ── Precompute softmax transition probabilities ──────────────────
#
# For each node u, transition prob to neighbor v is:
#   p(u→v) = exp(feat_u · feat_v) / Σ_w exp(feat_u · feat_w)
# This is the softmax formula given in Eq. (7) of the problem.
# Numerical stability: subtract max(dots) before exp (log-sum-exp).
print("\nPrecomputing softmax transition probabilities ...")
t0 = time.time()
trans = {}   # node → (nbrs: ndarray[int], probs: ndarray[float])
for node in gcc_nodes:
    nbrs = np.array(list(G_gcc.neighbors(node)), dtype=np.int64)
    if len(nbrs) == 0:
        trans[node] = (nbrs, np.array([], dtype=np.float64))
        continue
    dots  = text_np[nbrs] @ text_np[node]   # shape (|nbrs|,)
    dots -= dots.max()                        # numerical stability
    exp_d = np.exp(dots)
    trans[node] = (nbrs, exp_d / exp_d.sum())
print(f"  Done in {time.time()-t0:.2f}s")

# ── Random walk ──────────────────────────────────────────────────
WALK_LENGTH    = 100   # nodes visited per walk (including start)
WALKS_PER_SEED = 1000  # independent walks per seed node

def one_walk(start: int,
             seeds_arr: np.ndarray,
             alpha: float) -> np.ndarray:
    """
    One biased random walk of WALK_LENGTH steps starting from `start`.

    At each step:
      - With prob alpha    : teleport to a uniformly random class seed.
      - With prob 1-alpha  : move to a neighbor sampled by softmax prob.
    Dead-end nodes (no out-neighbors in GCC) always teleport.

    Returns a numpy int64 array of visited node IDs.
    """
    path = np.empty(WALK_LENGTH, dtype=np.int64)
    cur  = start
    for t in range(WALK_LENGTH):
        path[t] = cur
        if alpha > 0.0 and random.random() < alpha:
            cur = int(seeds_arr[random.randrange(len(seeds_arr))])
        else:
            nbrs, probs = trans[cur]
            if len(nbrs) == 0:
                cur = int(seeds_arr[random.randrange(len(seeds_arr))])
            else:
                cur = int(nbrs[np.random.choice(len(nbrs), p=probs)])
    return path


total_walks = sum(len(v) for v in seeds_by_class.values()) * WALKS_PER_SEED
print(f"\nWalk length    : {WALK_LENGTH} steps  (nodes visited per walk)")
print(f"Walks per seed : {WALKS_PER_SEED}")
print(f"Total walks    : {total_walks:,}  "
      f"(≈ {total_walks * WALK_LENGTH / 1e6:.1f}M node visits per α)")

# ── Run for α ∈ {0.0, 0.1, 0.2} ────────────────────────────────
ppr_summary = {}   # alpha → (acc, macro-f1, weighted-f1)

for alpha in [0.0, 0.1, 0.2]:
    print(f"\n{'─'*68}")
    print(f"α = {alpha}  (teleportation probability)")
    t0 = time.time()

    # visit_count[class_id] is a dict: node_id → visit count
    visit_count = [defaultdict(int) for _ in range(n_classes)]

    for c in range(n_classes):
        seeds_arr = np.array(seeds_by_class[c], dtype=np.int64)
        if len(seeds_arr) == 0:
            continue
        for seed in seeds_by_class[c]:
            for _ in range(WALKS_PER_SEED):
                path = one_walk(int(seed), seeds_arr, alpha)
                # Batch-count visits using np.unique (faster than
                # per-step dict update in a Python loop)
                uniq, cnts = np.unique(path, return_counts=True)
                for v, cnt in zip(uniq.tolist(), cnts.tolist()):
                    visit_count[c][v] += cnt

    # ── Classify GCC test nodes ──────────────────────────────────
    y_true, y_pred = [], []
    zero_visit = 0
    for node in gcc_test:
        counts = [visit_count[c][node] for c in range(n_classes)]
        if sum(counts) == 0:
            # Node never visited: fall back to training majority class
            y_pred.append(majority_class)
            zero_visit += 1
        else:
            y_pred.append(int(np.argmax(counts)))
        y_true.append(int(labels_np[node]))

    acc  = accuracy_score(y_true, y_pred)
    f1m  = f1_score(y_true, y_pred, average="macro")
    f1w  = f1_score(y_true, y_pred, average="weighted")

    print(f"  Elapsed time       : {time.time()-t0:.1f}s")
    print(f"  Zero-visit nodes   : {zero_visit} / {len(gcc_test)}")
    print(f"  Accuracy           : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Macro F1-score     : {f1m:.4f}")
    print(f"  Weighted F1-score  : {f1w:.4f}")
    print(f"\n  Per-class breakdown:")
    print(classification_report(
        y_true, y_pred,
        target_names=CLASS_NAMES,
        digits=4,
    ))
    ppr_summary[alpha] = (acc, f1m, f1w)


# ════════════════════════════════════════════════════════════════
# Final summary across all three questions
# ════════════════════════════════════════════════════════════════
print("═"*68)
print("FINAL SUMMARY")
print("═"*68)

print(f"\nQ23 — GCN")
print(f"  Architecture  : {best_rec['arch']}")
print(f"  # layers      : {best_rec['n_layers']}")
print(f"  dropout / lr / wd : "
      f"{best_rec['dropout']} / {best_rec['lr']} / {best_rec['wd']}")
print(f"  Test accuracy : {best_rec['test']*100:.2f}%")

print(f"\nQ24 — Node2Vec + Classifier")
print(f"  {'Feature set':<30} {'Best acc':>10}")
print(f"  {'─'*42}")
print(f"  {'Node2Vec only  (128-d)':<30} {acc_n2v*100:>9.2f}%")
print(f"  {'Text only     (1433-d)':<30} {acc_text*100:>9.2f}%")
print(f"  {'Combined      (1561-d)':<30} {acc_comb*100:>9.2f}%")
print(f"  Winner (single): {winner}")
print(f"  Best overall   : {best_acc*100:.2f}%  "
      f"({best_key[0].strip()} + {best_key[1]})")

print(f"\nQ25 — Personalized PageRank")
print(f"  {'α':>5} {'Accuracy':>10} {'Macro-F1':>10} {'Weighted-F1':>13}")
print(f"  {'─'*42}")
for alpha, (acc, f1m, f1w) in ppr_summary.items():
    print(f"  {alpha:>5.1f} {acc*100:>9.2f}%  {f1m:>10.4f}  {f1w:>13.4f}")

print("\n" + "═"*68)
print("DONE")
print("═"*68)

Device : cpu

Cora dataset:
  Nodes           : 2708
  Edges           : 10556
  Features/node   : 1433
  Classes         : 7
  Train/Val/Test  : 140 / 500 / 1000

════════════════════════════════════════════════════════════════════
Q23 — Graph Convolutional Network (GCN)
════════════════════════════════════════════════════════════════════

Grid search: 22 configurations, up to 500 epochs, early-stop patience = 100

  Architecture                    L  drop      lr      wd     val    test
  ────────────────────────────────────────────────────────────────────────
  [1433→16→7]                     2   0.5  0.0100 0.00050  0.7840  0.8060
  [1433→64→7]                     2   0.5  0.0100 0.00050  0.7920  0.8090
  [1433→128→7]                    2   0.5  0.0100 0.00050  0.7860  0.8030
  [1433→256→7]                    2   0.5  0.0100 0.00050  0.8060  0.8170
  [1433→64→7]                     2   0.3  0.0100 0.00050  0.7900  0.7980
  [1433→128→7]                    2   0.3  0.0100 0.00050  0.

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


  Done in 55.0s
  Embedding matrix: (2708, 128)  (nodes × dim)

Results — Accuracy / Macro-F1:

  Feature Set                  Classifier           Acc   Macro-F1
  ────────────────────────────────────────────────────────────────
  Node2Vec only  (128-d)       Random Forest     0.7050     0.6921
  Node2Vec only  (128-d)       SVM (RBF)         0.7450     0.7371
  Node2Vec only  (128-d)       MLP               0.7040     0.7017

  Text only     (1433-d)       Random Forest     0.5750     0.5656
  Text only     (1433-d)       SVM (RBF)         0.4540     0.4580
  Text only     (1433-d)       MLP               0.3500     0.3551

  Combined      (1561-d)       Random Forest     0.7290     0.7201
  Combined      (1561-d)       SVM (RBF)         0.6780     0.6904
  Combined      (1561-d)       MLP               0.4960     0.4868

────────────────────────────────────────────────────────────────────
Best acc — Node2Vec only : 0.7450  (74.50%)
Best acc — Text only     : 0.5750  (57.50%)
Winner 